# 01 · Infrastruttura Azure

Notebook per il deploy dell'infrastruttura della demo Foundry IQ con Azure CLI e Bicep, sul modello di [ElisaPiccin/wpc-foundryiq-demo](https://github.com/ElisaPiccin/wpc-foundryiq-demo). Ogni passaggio è in una cella.

| Passo | Cosa fa |
| --- | --- |
| 1. Configurazione | Ambiente, regione, nome del deployment ARM e parametri Bicep. |
| 2. Preflight | Login, tenant e subscription, utente demo, riuso dei valori di un deploy precedente, quote modelli. |
| 3. What-if | Anteprima delle modifiche (`az deployment sub what-if`). |
| 4. Deploy | `az deployment sub create` di [infra/main.bicep](../../infra/main.bicep), solo con `RUN_DEPLOY = True`. |
| 5. Output | Valori letti dagli output ARM; gli altri notebook li rileggono con lo stesso `DEPLOYMENT_NAME`. |

**Prerequisiti**: Azure CLI con Bicep (`az bicep install`), `az login`, ruolo *Owner* (o *Contributor* + *User Access Administrator*) sulla subscription, ambiente Python con `pip install -r notebooks/requirements.txt`.

**Sequenza**: `01-infrastruttura` → `02-dati-identita-app` → `03-foundry-iq-agenti`, poi i notebook in [demo](../demo/).

In [ ]:
# 1 · Configurazione: modifica solo questa cella.
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: serve infra/main.bicep.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from notebook_env import load_azd_env

LOAD_AZD_ENV = False                 # True = precompila con l'ambiente azd
AZD_ENVIRONMENT_NAME = ""             # vuoto = ambiente selezionato; oppure un nome esplicito
azd_values = load_azd_env(AZD_ENVIRONMENT_NAME, root=ROOT, override=True) if LOAD_AZD_ENV else {}
if LOAD_AZD_ENV:
    missing = [k for k in ("AZURE_ENV_NAME", "AZURE_SUBSCRIPTION_ID", "AZURE_TENANT_ID") if not azd_values.get(k)]
    if missing:
        raise RuntimeError(f"Valori mancanti nell'ambiente azd: {', '.join(missing)}.")

SUBSCRIPTION_ID = azd_values.get("AZURE_SUBSCRIPTION_ID", "")  # vuoto = subscription attiva
ENVIRONMENT_NAME = azd_values.get("AZURE_ENV_NAME") or "wpc-demo"
LOCATION = azd_values.get("AZURE_LOCATION") or "swedencentral"
RESOURCE_GROUP_NAME = azd_values.get("AZURE_RESOURCE_GROUP", "")  # vuoto = rg-<ENVIRONMENT_NAME>
DEPLOYMENT_NAME = azd_values.get("AZURE_DEPLOYMENT_NAME") or f"wpc-foundryiq-{ENVIRONMENT_NAME}"

# Parametri opzionali di infra/main.bicep (i valori obbligatori sono calcolati nel preflight).
BICEP_PARAMETERS = {
    "appServiceSkuName": "B1",       # F1 non supporta il private endpoint SQL
    "chatDeploymentSku": "DataZoneStandard",
    "chatCapacity": 200,
    "embeddingCapacity": 20,
    "gpt52Capacity": 200,
    "sqlPrivateEndpoint": True,      # False = SQL pubblico, il notebook 02 carica i dati dal client
    "sqlUseFreeLimit": True,
}

RUN_WHAT_IF = True                   # anteprima delle modifiche
RUN_DEPLOY = False                   # True = crea/aggiorna le risorse (20-40 minuti, costi Azure)

In [ ]:
# Bootstrap: Azure CLI, login, subscription.
import json
import os
import secrets
import shutil
import subprocess
import tempfile
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: serve infra/main.bicep.")
TEMPLATE = ROOT / "infra" / "main.bicep"
AZ = shutil.which("az")
if AZ is None:
    raise RuntimeError("Azure CLI non trovata nel PATH: installala, esegui 'az login' e riavvia il kernel.")


def az(*args, output=True, check=True):
    """Esegue Azure CLI. Con output=True restituisce il JSON decodificato; con check=False None in caso di errore."""
    command = [AZ, *args, "--only-show-errors"] + (["--output", "json"] if output else [])
    proc = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        if check:
            raise RuntimeError(f"az {' '.join(args[:3])} fallito: {proc.stderr.strip()[-2000:]}")
        return None
    text = proc.stdout.strip()
    return (json.loads(text) if text else None) if output else text


if SUBSCRIPTION_ID:
    az("account", "set", "--subscription", SUBSCRIPTION_ID, output=False)
account = az("account", "show", check=False)
if not account:
    raise RuntimeError("Azure CLI non autenticata: esegui 'az login' (eventualmente con --tenant).")
if azd_values.get("AZURE_TENANT_ID") and azd_values["AZURE_TENANT_ID"].casefold() != account["tenantId"].casefold():
    raise RuntimeError("Il tenant dell'ambiente azd non corrisponde alla subscription selezionata in Azure CLI.")
SUBSCRIPTION_ID, TENANT_ID = account["id"], account["tenantId"]
print(f"Subscription: {account['name']} ({SUBSCRIPTION_ID}), tenant {TENANT_ID}")
print(f"Azure CLI {az('version')['azure-cli']}; Bicep: {az('bicep', 'version', output=False, check=False) or 'non installato (az bicep install)'}")

## 2 · Preflight

Ricava i parametri obbligatori del Bicep senza file locali:

- **utente demo** (`demoUserObjectId`, `sqlAdminLogin`): l'utente connesso ad Azure CLI, che riceve i ruoli dati su Search, Storage e Foundry;
- **chiave MCP** (`mcpApiKey`): riletta dalle impostazioni dell'App Service MCP se il deployment esiste già, altrimenti generata; resta solo in memoria e non viene stampata;
- **client ID della chat** (`chatAuthClientId`): l'app registration `wpc-foundryiq-chat-<ambiente>` creata dal notebook 02, se presente, così un nuovo deploy non disattiva il login Entra;
- **quote** dei modelli nella regione (solo avviso).

In [ ]:
demo_user = az("ad", "signed-in-user", "show", check=False)
if not demo_user:
    raise RuntimeError("Serve un account utente Entra (non un service principal) per i ruoli della demo e l'admin SQL.")
print(f"Utente demo: {demo_user['userPrincipalName']} ({demo_user['id']})")

sku = str(BICEP_PARAMETERS.get("appServiceSkuName", "B1")).upper()
if sku == "F1" and BICEP_PARAMETERS.get("sqlPrivateEndpoint", True):
    raise ValueError("App Service F1 non supporta la VNet integration: usa B1 o imposta sqlPrivateEndpoint=False.")

previous = az("deployment", "sub", "show", "--name", DEPLOYMENT_NAME, check=False)
previous_outputs = {k.upper(): v["value"] for k, v in ((previous or {}).get("properties", {}).get("outputs") or {}).items()}
mcp_api_key = None
if previous_outputs.get("MCP_APP_NAME"):
    settings = az("webapp", "config", "appsettings", "list", "--resource-group", previous_outputs["AZURE_RESOURCE_GROUP"],
                  "--name", previous_outputs["MCP_APP_NAME"], check=False) or []
    mcp_api_key = next((s["value"] for s in settings if s["name"] == "MCP_API_KEY"), None)
print("Chiave MCP:", "riusata dall'App Service esistente" if mcp_api_key else "nuova (32 byte casuali)")
mcp_api_key = mcp_api_key or secrets.token_hex(32)

chat_apps = az("ad", "app", "list", "--display-name", f"wpc-foundryiq-chat-{ENVIRONMENT_NAME}", check=False) or []
chat_client_id = chat_apps[0]["appId"] if chat_apps else ""
print("App registration chat:", chat_client_id or "non ancora creata (la crea il notebook 02)")

import pandas as pd

chat_sku = BICEP_PARAMETERS.get("chatDeploymentSku", "DataZoneStandard")
required_quota = {
    f"OpenAI.{chat_sku}.gpt-5-mini": BICEP_PARAMETERS.get("chatCapacity", 200),
    "OpenAI.Standard.text-embedding-3-large": BICEP_PARAMETERS.get("embeddingCapacity", 20),
    "OpenAI.GlobalStandard.gpt-5.2": BICEP_PARAMETERS.get("gpt52Capacity", 200),
}
usages = {u["name"]["value"]: u for u in az("cognitiveservices", "usage", "list", "--location", LOCATION, check=False) or []}
rows = []
for name, needed in required_quota.items():
    usage = usages.get(name)
    free = int(usage["limit"] - usage["currentValue"]) if usage else None
    rows.append({"quota": name, "richiesta (K TPM)": needed, "disponibile": free,
                 "esito": "?" if free is None else "OK" if free >= needed else "INSUFFICIENTE"})
display(pd.DataFrame(rows))
if any(r["esito"] == "INSUFFICIENTE" for r in rows):
    print("Attenzione: quota insufficiente. Riduci le capacità in BICEP_PARAMETERS o cambia LOCATION. "
          "Un deployment esistente dello stesso modello conta già nella quota usata.")

## 3 · Parametri e what-if

I parametri sono scritti in un file temporaneo fuori dal repository (contiene la chiave MCP) e cancellato subito dopo l'uso.

In [ ]:
deployment_parameters = {
    "environmentName": ENVIRONMENT_NAME,
    "location": LOCATION,
    "resourceGroupName": RESOURCE_GROUP_NAME,
    "demoUserObjectId": demo_user["id"],
    "demoUserPrincipalType": "User",
    "sqlAdminLogin": demo_user["userPrincipalName"],
    "chatAuthClientId": chat_client_id,
    **BICEP_PARAMETERS,
}


def az_with_parameters(*args, output=True):
    handle, path = tempfile.mkstemp(suffix=".parameters.json")
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as stream:
            values = {**deployment_parameters, "mcpApiKey": mcp_api_key}
            json.dump({"$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
                       "contentVersion": "1.0.0.0", "parameters": {k: {"value": v} for k, v in values.items()}}, stream)
        return az(*args, "--name", DEPLOYMENT_NAME, "--location", LOCATION, "--template-file", str(TEMPLATE),
                  "--parameters", f"@{path}", output=output)
    finally:
        os.remove(path)


display(pd.DataFrame([{"parametro": k, "valore": v} for k, v in deployment_parameters.items()]))
if RUN_WHAT_IF:
    print(az_with_parameters("deployment", "sub", "what-if", "--result-format", "ResourceIdOnly", output=False))
else:
    print("RUN_WHAT_IF=False: anteprima saltata.")

## 4 · Deploy

Crea o aggiorna resource group, Foundry (progetto e modelli), Azure AI Search, Storage (manuali e ADLS ACL), Azure SQL, App Service MCP e chat, monitoraggio e ruoli. È idempotente: rieseguirlo applica solo le differenze.

In [ ]:
if RUN_DEPLOY:
    print(f"Deploy '{DEPLOYMENT_NAME}' in corso (20-40 minuti)...")
    result = az_with_parameters("deployment", "sub", "create")
    print("Stato:", result["properties"]["provisioningState"])
else:
    print("RUN_DEPLOY=False: nessuna modifica. Imposta RUN_DEPLOY = True e riesegui la cella.")

## 5 · Output del deployment

Gli altri notebook leggono questi valori con `az deployment sub show --name <DEPLOYMENT_NAME>`: non serve salvarli in file locali. Gli output non contengono segreti.

In [ ]:
deployment = az("deployment", "sub", "show", "--name", DEPLOYMENT_NAME, check=False)
if not deployment:
    print(f"Deployment '{DEPLOYMENT_NAME}' non ancora presente.")
else:
    outputs = {k.upper(): v["value"] for k, v in (deployment["properties"].get("outputs") or {}).items()}
    print("Stato:", deployment["properties"]["provisioningState"])
    display(pd.DataFrame([{"output": k, "valore": v} for k, v in sorted(outputs.items())]))
    print("Prossimo passo: notebooks/deploy/02-dati-identita-app.ipynb con lo stesso ENVIRONMENT_NAME.")
mcp_api_key = None  # non lasciare la chiave nello stato del kernel